## Utils

In [ ]:
import os
from xml.etree.ElementTree import parse

def get_xml_files(dir):
    """
    Traverse the subtype folders and collect paths to XML files.
    :param base_folder: Root folder containing subtype folders.
    :return: List of (subtype, xml_file_path) tuples.
    """
    xml_files = []

    for subtype in os.listdir(dir):
        subtype_folder = os.path.join(dir, subtype)
        if not os.path.isdir(subtype_folder):
            continue

        xml_folder = os.path.join(subtype_folder, f"{subtype}_xml")
        if not os.path.exists(xml_folder):
            continue

        for file in os.listdir(xml_folder):
            if file.endswith(".xml"):
                xml_files.append((subtype, os.path.join(xml_folder, file)))

    return xml_files

def compute_polygon_area(coords):
    """
    Compute the surface area of a polygon using the Shoelace formula.
    :param coords: List of (X, Y) tuples.
    :return: Surface area.
    """
    n = len(coords)
    area = 0
    for i in range(n):
        x1, y1 = coords[i]
        x2, y2 = coords[(i + 1) % n]
        area += x1 * y2 - x2 * y1
    return abs(area) / 2

def parse_and_check(xml_file_path):
    """
    Parse an XML file, check annotations, and compute surfaces.
    :param xml_file_path: Path to the XML file.
    :return: List of annotation details (subtype, annotation name, type, group, area).
    """
    results = []
    tree = parse(xml_file_path)
    root = tree.getroot()

    annotations = root.find("Annotations")
    for annotation in annotations.findall("Annotation"):
        annotation_type = annotation.get("Type")
        part_of_group = annotation.get("PartOfGroup")

        coords = []
        for coord in annotation.find("Coordinates").findall("Coordinate"):
            coords.append((float(coord.get("X")), float(coord.get("Y"))))

        area = compute_polygon_area(coords)

        results.append({
            "Annotation Name": annotation.get("Name"),
            "Type": annotation_type,
            "Group": part_of_group,
            "Surface Area": area
        })
    return results

## XML Parser

In [ ]:
import platform
import json
import pandas as pd

OS_MODE = platform.system()

try:
    with open("_info/config.json", 'r') as file:
        conf = json.load(file)
except:
    raise FileNotFoundError("Couldn't Load Config File. Check if the `config.json` file exists under `_info` directory.")

xml_files = get_xml_files(conf['wsis_dir'])
final_results = []

for subtype, xml_file in xml_files:
    print(f"Checking {subtype}, {xml_file} ...")
    annotation_results = parse_and_check(xml_file)
    for result in annotation_results:
        result["Subtype"] = subtype
        result["File"] = os.path.basename(xml_file)
        final_results.append(result)

annot_dfs = pd.DataFrame(final_results)

## Validating Annotations

In [ ]:
annot_dfs.head()

### Check Type of Annotations

Should be 'Polygon' or 'Spline'

In [ ]:
annot_dfs['Type'].value_counts()

### Check the Group Annotations
Should be 'tumor' or 'normal'

In [ ]:
annot_dfs['Group'].value_counts()

### Get total Surface Area of Each Label

In [ ]:
grouped_areas = annot_dfs.groupby("Group")["Surface Area"].sum().reset_index()
grouped_areas.columns = ["Group", "Total Surface Area"]
grouped_areas